# Data Analysis and Exploration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# anchor everything relative to the notebook's location
DATA_DIR = Path("../data/allData")

# confirm it resolves correctly
print(DATA_DIR.resolve())
print("Exists:", DATA_DIR.exists())

# list the trading date folders
trading_days = sorted(DATA_DIR.iterdir())
print(f"\n{len(trading_days)} trading days found:")
for d in trading_days:
    print(" ", d.name)

## Section 1 — Single File Inspection

Before doing anything at scale, we look at one file with our own eyes.
The goal is to answer five questions that every subsequent decision depends on:
- Does the file have a header row, or are the columns unnamed?
- What does the Time column look like — HH:MM:SS, Unix timestamp, something else?
- How many rows is a single day? (~75 = 1-min bars, thousands = raw ticks)
- Is Volume per-trade or cumulative through the day?
- Are prices option premiums (tens/hundreds) or index levels (tens of thousands)?

We look at both an options file and a futures file before drawing any conclusions.

In [ ]:
day = trading_days[0]
print(f"Analyzing data for {day}...")

options_dir = day / "Options"
futures_dir = day / "Futures (Continuous)"

options_files = sorted(options_dir.iterdir())
futures_files = sorted(futures_dir.iterdir())

print(f"\n{len(options_files)} options files found")
print(f"{len(futures_files)} futures files found")

print("\n Top 5 options files:")
for f in options_files[:5]:
    print(" ", f.name)

print("\n Top 5 futures files:")
for f in futures_files[:5]:
    print(" ", f.name)

In [ ]:
sample_option_file = options_files[0]
print("Reading:", sample_option_file.name)

df_opt = pd.read_csv(sample_option_file)

print("\nShape:", df_opt.shape)
print("\nColumn names:", df_opt.columns.tolist())
print("\nDtypes:\n", df_opt.dtypes)
print("\nFirst 10 rows:")
df_opt.head(10)

In [ ]:
sample_futures_file = futures_files[0]
print("Reading:", sample_futures_file.name)

df_fut = pd.read_csv(sample_futures_file)

print("\nShape:", df_fut.shape)
print("\nColumn names:", df_fut.columns.tolist())
print("\nDtypes:\n", df_fut.dtypes)
print("\nFirst 10 rows:")
df_fut.head(10)

### Critical finding: no header row

The column names printed above (`'20221101'`, `'09:15:02'`, `'1.40'`) are
not headers — they are the **first row of actual data** being misread as
column names. Every file in this dataset has the same issue.

Per the assignment spec, the five columns are: Date, Time, Price, Volume, Open Interest.
We must pass `header=None, names=COLUMNS` on every read, or we silently lose
the first data point of every instrument every day.

We also confirmed this is **tick data** (4779 rows for one file in a 375-minute 
session), not 1-minute bars. Multiple trades can happen within the same second.
This means we cannot directly join options against futures on raw timestamps —
we need to resample to a common 1-minute grid first.

In [ ]:
COLUMNS = ['date', 'time', 'price', 'volume', 'open_interest']

df_opt = pd.read_csv(options_files[0], header=None, names=COLUMNS)
df_fut = pd.read_csv(futures_files[0], header=None, names=COLUMNS)

print("=== OPTIONS FILE ===")
print("Shape:", df_opt.shape)
print(df_opt.head(10))
print("\nPrice range:", df_opt['price'].min(), "to", df_opt['price'].max())
print("Volume range:", df_opt['volume'].min(), "to", df_opt['volume'].max())
print("Unique timestamps:", df_opt['time'].nunique())
print("Time range:", df_opt['time'].min(), "to", df_opt['time'].max())

print("\n=== FUTURES FILE (BANKNIFTY-I) ===")
print("Shape:", df_fut.shape)
print(df_fut.head(10))
print("\nPrice range:", df_fut['price'].min(), "to", df_fut['price'].max())
print("Unique timestamps:", df_fut['time'].nunique())
print("Time range:", df_fut['time'].min(), "to", df_fut['time'].max())

### What the corrected read tells us

- **Options price range**: 1.15 to 1.90 — these are option *premiums* in rupees,
  not index levels. Makes sense: this is a deep OTM put on BANKNIFTY, nearly worthless.
- **Futures price range**: 41,330 to 41,825 — this IS the index level. BANKNIFTY spot.
- **Time range**: Options run 09:15:02 to 15:29:57. Futures run 09:15:00 to 15:30:00.
  The NSE session is 09:15–15:30, so full coverage confirmed.
- **Unique timestamps**: Options has 4628 unique timestamps for 4780 rows — so some
  seconds have multiple trades. Futures has 11462 unique timestamps for 11465 rows —
  same pattern, just more liquid.

In [ ]:
# does OI change through the day?
print("=== OI in Options file ===")
print("Unique OI values:", df_opt['open_interest'].nunique())
print("OI value counts (top 5):\n", df_opt['open_interest'].value_counts().head())

print("\n=== OI in Futures file ===")
print("Unique OI values:", df_fut['open_interest'].nunique())
print("OI value counts (top 5):\n", df_fut['open_interest'].value_counts().head())

### Open Interest behaviour

OI has **104 unique values** in the options file and **119 in futures** — it is
updating throughout the day with each trade, not just reported once at end of day.

Two implications:
1. When we resample tick data to 1-minute bars, OI should be aggregated with `.last()`
   (it's a running total) while Volume should use `.sum()` (it's per-trade lots).
   These are different aggregation rules and they matter.
2. OI *could* be a signal for the strategy (rising OI while in a position = more
   participants entering). We note this but don't act on it yet.

In [ ]:
df_opt['datetime'] = pd.to_datetime(
    df_opt['date'].astype(str) + ' ' + df_opt['time'],
    format='%Y%m%d %H:%M:%S'
)
df_opt = df_opt.set_index('datetime')

# resample to 1-minute OHLCV
opt_1min = df_opt['price'].resample('1min').ohlcv() if hasattr(df_opt['price'].resample('1min'), 'ohlcv') else \
    df_opt['price'].resample('1min').agg(['first', 'max', 'min', 'last'])

opt_vol_1min = df_opt['volume'].resample('1min').sum()

print("1-minute bars shape:", opt_1min.shape)
print("\nFirst 10 bars:")
print(opt_1min.head(10))
print("\nVolume per minute (first 10):")
print(opt_vol_1min.head(10))

# how many minutes have zero trades?
print(f"\nMinutes with no trades: {opt_1min['last'].isna().sum()} out of {len(opt_1min)}")

### Resampling tick data to 1-minute bars

After resampling, we get exactly **375 bars** — matching the 09:15–15:29 NSE session
perfectly. This will be our uniform time grid for the entire backtest.

Key finding: **68 out of 375 minutes (18%) have no trades** for this one options file.
But this file is BANKNIFTY at strike 30,500 when BANKNIFTY spot was ~41,650 — 
that's 26% out of the money, barely anyone touches it. Near-ATM strikes will be
far more liquid. We'll quantify this properly in Section 4 (ATM feasibility).

Volume spikes hard at the open (250,601 lots in the first minute), which is typical.
Entering a straddle at 09:15 means filling during the most volatile, widest-spread
period of the day. We'll document our entry-timing assumption when we get to the
backtest engine.

## Section 2 — Universe Inventory

Now we zoom out from one file to the entire dataset: all 21 days, all underliers,
all instruments. The output of this section is a single DataFrame — the **universe
table** — with one row per (date, instrument). Every subsequent question about
expiries, strikes, and straddle feasibility is just a query on this table.

### The filename parser

Before scanning 31,000+ files, we need a reliable way to extract structured 
information from each filename. The naming convention is:

    {UNDERLIER}{YYMMDD}{STRIKE}{CE|PE}.csv

The underlier is variable length (NIFTY=5, BANKNIFTY=9, FINNIFTY=8 chars), so
we cannot slice fixed character positions — it would break on BANKNIFTY. Instead
we anchor the parse on what IS fixed: expiry is always 6 digits, option type is
always the last 2 characters. A single regex handles all cases cleanly.

In [ ]:
import re

INSTRUMENT_RE = re.compile(r"^([A-Z]+)(\d{6})(\d+)(CE|PE)$")

def parse_filename(filename: str) -> dict | None:
    """
    Parse an options filename into its components.
    Returns None (not raises) so we can handle bad files gracefully during scan.
    """
    name = filename.replace(".csv", "")
    m = INSTRUMENT_RE.match(name)
    if not m:
        return None
    underlier, expiry, strike, option_type = m.groups()
    return {
        "underlier": underlier,
        "expiry": expiry,
        "strike": int(strike),
        "option_type": option_type,
    }

# test against the three spec examples before touching real data
tests = [
    ("NIFTY22110314550PE.csv",      ("NIFTY",     "221103", 14550, "PE")),
    ("BANKNIFTY22112443200CE.csv",  ("BANKNIFTY", "221124", 43200, "CE")),
    ("FINNIFTY22110719500CE.csv",   ("FINNIFTY",  "221107", 19500, "CE")),
]

print("Parser tests:")
for filename, (exp_u, exp_e, exp_s, exp_t) in tests:
    result = parse_filename(filename)
    if result is None:
        print(f"  ✗  {filename} (failed to parse)")
        continue
    ok = (
        result["underlier"] == exp_u and
        result["expiry"] == exp_e and
        result["strike"] == exp_s and
        result["option_type"] == exp_t
    )
    print(f"  {'✓' if ok else '✗'}  {filename}")

All three spec examples parse correctly, including BANKNIFTY which would break 
any naive fixed-position slicer. The parser is now trustworthy for the full scan.

In [ ]:
records = []

for day_dir in sorted(DATA_DIR.iterdir()):
    date_str = day_dir.name.replace("NSE_", "")   # "20221101"
    options_dir = day_dir / "Options"
    
    if not options_dir.exists():
        print(f"WARNING: No Options folder in {day_dir.name}")
        continue
    
    for f in options_dir.iterdir():
        parsed = parse_filename(f.name)
        if parsed is None:
            print(f"  UNPARSEABLE: {f.name}")
            continue
        parsed["date"] = date_str
        parsed["filename"] = f.name
        records.append(parsed)

universe = pd.DataFrame(records)

# convert date and expiry to proper types
universe["date"] = pd.to_datetime(universe["date"], format="%Y%m%d")
universe["expiry_date"] = pd.to_datetime("20" + universe["expiry"], format="%Y%m%d")

print(f"Total instrument-days: {len(universe):,}")
print(f"\nUnique underliers: {universe['underlier'].unique()}")
print(f"Unique dates: {universe['date'].nunique()}")
print(f"\nRows per underlier:")
print(universe['underlier'].value_counts())

### 4 underliers, not 3

The spec mentions NIFTY, BANKNIFTY, FINNIFTY — but the data also contains **MIDCPNIFTY**.

- NIFTY: 16,356 instrument-days (dominant)
- BANKNIFTY: 12,979 instrument-days
- FINNIFTY: 1,962 instrument-days
- MIDCPNIFTY: **22 instrument-days** (negligible)

MIDCPNIFTY only appears on 5 specific dates and only has 0–8 files per day.
As we'll see in the expiry structure, it only appears on its own expiry day.

**Decision**: MIDCPNIFTY is excluded from the backtest. Focus is on NIFTY and
BANKNIFTY. FINNIFTY is kept but secondary.

In [ ]:
daily_counts = (
    universe
    .groupby(["date", "underlier"])
    .size()
    .unstack(fill_value=0)
)

print("Instruments per day per underlier:")
print(daily_counts.to_string())
print(f"\nMin per day: {daily_counts.min().to_dict()}")
print(f"Max per day: {daily_counts.max().to_dict()}")


BANKNIFTY ranges from 537 to 724 files per day. NIFTY from 708 to 864.
This happens because exchanges list more strike prices when volatility is
higher (more strikes become "relevant" and attract open interest). The counts
varying is expected and not a data quality issue.

What matters for the backtest: the *near-ATM* strikes are always present.
The count variation happens in the deep OTM tail, which we don't trade.

In [ ]:
# what expiries are live on each date, per underlier?
expiry_by_underlier = (
    universe
    .groupby(["underlier", "date"])["expiry_date"]
    .unique()
    .apply(sorted)
)

print("Expiries live per underlier per date (first 5 dates):\n")
for underlier in universe["underlier"].unique():
    print(f"=== {underlier} ===")
    subset = expiry_by_underlier[underlier]
    for date, expiries in subset.head(5).items():
        print(f"  {date.date()}  →  {[str(e.date()) for e in expiries]}")
    print()

### Expiry structure

**NIFTY and BANKNIFTY**: Weekly expiries every **Thursday** (Nov 3, 10, 17, 24, Dec 1...).
Plus a long tail of monthly, quarterly, and annual contracts stretching to Dec 2026.
The long-dated contracts are illiquid noise — a rolling short-term straddle only 
ever trades the **nearest upcoming Thursday expiry** (the "front week").

**FINNIFTY**: Weekly expiries every **Tuesday** (Nov 7, 15, 22, 29...). 
Notable: Nov 1 shows an expiry dated Nov 1 itself — options expiring *today* still
trade until 15:30. Also, Nov 8 (a Tuesday) was Guru Nanak Jayanti — a market holiday.
The exchange moved that expiry to Nov 7 (Monday), the previous trading day. 
This kind of holiday-adjusted expiry is real market behaviour and must be handled
by reading the *actual* expiry dates from the data, not assuming fixed weekdays.

**Rolling rule that falls out of this**: When today is an expiry day for an underlier,
the front-week contract expires at close. The next trading day, you roll to the 
*new* front-week expiry (next Thursday for NIFTY/BANKNIFTY, next Tuesday for FINNIFTY).

In [ ]:
# pivot: for each (date, underlier, expiry, strike) — is CE present? PE present?
pair_check = (
    universe
    .groupby(["date", "underlier", "expiry_date", "strike"])["option_type"]
    .apply(set)
    .reset_index()
)

pair_check["has_CE"] = pair_check["option_type"].apply(lambda x: "CE" in x)
pair_check["has_PE"] = pair_check["option_type"].apply(lambda x: "PE" in x)
pair_check["is_pair"] = pair_check["has_CE"] & pair_check["has_PE"]

total = len(pair_check)
paired = pair_check["is_pair"].sum()
ce_only = (~pair_check["has_PE"] & pair_check["has_CE"]).sum()
pe_only = (~pair_check["has_CE"] & pair_check["has_PE"]).sum()

print(f"Total (date, underlier, expiry, strike) combinations: {total:,}")
print(f"  Full pairs (CE + PE): {paired:,}  ({100*paired/total:.1f}%)")
print(f"  CE only:              {ce_only:,}  ({100*ce_only/total:.1f}%)")
print(f"  PE only:              {pe_only:,}  ({100*pe_only/total:.1f}%)")

# if there are orphans, where are they?
if ce_only + pe_only > 0:
    orphans = pair_check[~pair_check["is_pair"]]
    print("\nSample orphans:")
    print(orphans.head(10)[["date", "underlier", "expiry_date", "strike", "has_CE", "has_PE"]])

Only 52% of (date, underlier, expiry, strike) combinations have both a CE and PE file.
The rest are one-legged. But look at the orphan sample: all are PE-only at strikes
30,500–35,000 for BANKNIFTY when spot was ~41,650. Those strikes are 15–26% below spot —
deep OTM puts. Nobody trades deep OTM *calls* at the same strike, so the exchange
simply doesn't list them. The asymmetry is structural, not a data quality issue.

The mirror happens on the upside: very high strike calls will have no matching puts.

**For the straddle, this doesn't matter** — we only need paired strikes *near spot*.
Near-ATM strikes (within ~2% of the futures price) should have full CE+PE pairs.

## Section 3 — ATM Feasibility

The strategy operates at 1-second granularity:
> "At every 1 sec, select the strike closest to the futures price. 
>  Buy both CE and PE. Hold until the futures price changes the strike."

This means we do NOT resample to 1-minute bars. Instead, at each second we need
the **last known price** for futures and options — i.e., the most recent tick
that occurred at or before that second. This is a forward-fill operation.

Five questions to answer:
1. What is the front-week expiry for each date and underlier?
2. What does the futures price look like at 1-second resolution?
3. How often does the nearest strike change? (determines roll frequency and costs)
4. When does the first option tick appear for the ATM strike? (determines entry lag)
5. Across all 21 days, do both CE and PE legs exist for the opening ATM strike?

### Step 1: Front-week expiry

"Instruments expiring closest to the current date" = the minimum expiry_date
that is >= the trading date. On expiry day itself, the expiring contract still
trades until 15:30 so it remains the front-week contract for that whole day.

In [ ]:
# front-week expiry = nearest upcoming expiry on each trading date
live = universe[universe["expiry_date"] >= universe["date"]].copy()

front_week = (
    live
    .groupby(["date", "underlier"])["expiry_date"]
    .min()
    .reset_index()
    .rename(columns={"expiry_date": "front_expiry"})
)

# display for NIFTY and BANKNIFTY only
pivot = (
    front_week[front_week["underlier"].isin(["NIFTY", "BANKNIFTY"])]
    .pivot(index="date", columns="underlier", values="front_expiry")
)
pivot.index = pivot.index.strftime("%Y-%m-%d (%a)")
for col in pivot.columns:
    pivot[col] = pivot[col].dt.strftime("%Y-%m-%d")

print("Front-week expiry per trading date:\n")
print(pivot.to_string())

### Finding: Front-week rolls happen cleanly on Thursdays

The table confirms the rolling calendar:
- NIFTY and BANKNIFTY both use Thursday weekly expiries
- On Nov 3 (Thursday), front expiry was Nov 3 all week. On Nov 4 it becomes Nov 10.
- The roll is automatic: always pick `min(expiry_date >= trading_date)`

No special handling needed. The logic is the same every week.

### Step 2: Futures price at 1-second resolution

The strategy picks a strike "at every 1 sec" based on the current futures price.
We need a price at every second of the session (09:15:00 → 15:29:59 = 22,500 seconds).

The approach: take raw futures ticks, resample to 1-second using last() to handle
multiple trades within the same second, then forward-fill onto the full second grid.
Forward-fill means we always use the most recently observed price — no look-ahead.

A NaN in the final series would mean no futures trade has occurred yet since open.
For a liquid contract like BANKNIFTY-I this should be zero or near-zero.

In [ ]:
def load_futures_1sec(day_dir: Path, underlier: str) -> pd.Series:
    """
    Load futures tick data and produce a forward-filled 1-second price series
    for the full NSE session (09:15:00 to 15:29:59).
    """
    path = day_dir / "Futures (Continuous)" / f"{underlier}-I.csv"
    df = pd.read_csv(path, header=None, names=COLUMNS)
    df["datetime"] = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"],
        format="%Y%m%d %H:%M:%S"
    )
    df = df.set_index("datetime").sort_index()

    # build the complete 1-second grid for this date
    date_str = day_dir.name.replace("NSE_", "")
    session_start = pd.Timestamp(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]} 09:15:00")
    session_end   = pd.Timestamp(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]} 15:29:59")
    full_seconds  = pd.date_range(session_start, session_end, freq="1s")

    # resample to 1-sec (last trade per second), then ffill gaps
    price_1sec = df["price"].resample("1s").last()
    return price_1sec.reindex(full_seconds).ffill()

sample_dir = trading_days[0]
fut_1sec = load_futures_1sec(sample_dir, "BANKNIFTY")

print(f"Total seconds in session : {len(fut_1sec):,}  (expect 22,500)")
print(f"Seconds with NaN price   : {fut_1sec.isna().sum()}")
print(f"Price at 09:15:00        : {fut_1sec.iloc[0]:.2f}")
print(f"Price at 09:30:00        : {fut_1sec.at[pd.Timestamp('2022-11-01 09:30:00')]:.2f}")
print(f"Price at 15:29:59        : {fut_1sec.iloc[-1]:.2f}")
print(f"\nFirst 5 seconds:")
print(fut_1sec.head())

### Finding: Futures price is perfectly clean at 1-second resolution

- 22,500 seconds in session ✓
- 0 NaN gaps — the futures price is never missing ✓
- BANKNIFTY-I is extremely liquid: a trade occurs nearly every second

This means the strategy's ATM reference price is always available.
We never have to decide "what do we do when we don't know the futures price."
That problem simply doesn't exist in this dataset.

### Step 3: Roll frequency

Every time the nearest strike changes, the strategy closes the current straddle
and opens a new one. That's two sells and two buys — real friction each time.

We compute the ATM strike at every second of one sample day and count how many
times it changes. This tells us:
- How active this strategy is (rolls per day)
- How sensitive PnL will be to transaction cost assumptions

We use searchsorted (binary search) rather than brute-force argmin to find
the nearest strike efficiently across all 22,500 seconds at once.

In [ ]:
def find_atm_vectorized(prices: np.ndarray, strikes_sorted: np.ndarray) -> np.ndarray:
    """
    For each price, find the nearest value in strikes_sorted.
    Uses binary search — fast even for 22,500 prices × 700 strikes.
    """
    idx = np.searchsorted(strikes_sorted, prices)
    idx = np.clip(idx, 1, len(strikes_sorted) - 1)
    left  = strikes_sorted[idx - 1]
    right = strikes_sorted[idx]
    # pick whichever side is closer; ties go to the lower strike
    return np.where(prices - left <= right - prices, left, right)


# get available strikes for front-week BANKNIFTY on Nov 1
date = pd.Timestamp(sample_dir.name.replace("NSE_", ""))
front_exp = front_week[
    (front_week["date"] == date) & (front_week["underlier"] == "BANKNIFTY")
]["front_expiry"].iloc[0]

day_strikes = universe[
    (universe["date"] == date) &
    (universe["underlier"] == "BANKNIFTY") &
    (universe["expiry_date"] == front_exp)
]["strike"].unique()
strikes_arr = np.array(sorted(day_strikes))

# ATM strike at every second
prices = fut_1sec.values
atm_per_second = pd.Series(
    find_atm_vectorized(prices, strikes_arr),
    index=fut_1sec.index
)

# a "roll" happens whenever the ATM strike changes
rolls = (atm_per_second != atm_per_second.shift(1)).sum() - 1  # exclude first entry

print(f"BANKNIFTY {date.date()} — Roll frequency:")
print(f"  Total seconds in session : {len(atm_per_second):,}")
print(f"  ATM strike changes (rolls): {rolls}")
print(f"  Unique ATM strikes visited: {atm_per_second.nunique()}")
print(f"\nATM strike over first 30 seconds:")
print(atm_per_second.head(30).to_string())

# show where rolls happen (first 10)
roll_times = atm_per_second[atm_per_second != atm_per_second.shift(1)].iloc[1:11]
print(f"\nFirst 10 rolls:")
for ts, strike in roll_times.items():
    prev = atm_per_second[atm_per_second.index < ts].iloc[-1]
    print(f"  {ts.time()}  {int(prev)} → {int(strike)}")

### Finding: 237 rolls in one day

On Nov 1, BANKNIFTY rolled its ATM strike 237 times. The price spent most of the
day bouncing between strikes 41,600 and 41,700 — only 6 unique strikes were ever
visited. The strategy was not calmly holding a straddle; it was ping-ponging between
two adjacent strikes, sometimes multiple times per minute.

Each roll = sell CE + sell PE + buy CE + buy PE = 4 transactions.
237 rolls × 4 = ~948 transactions in one day on one underlier alone.

**Implication**: Transaction costs are not a minor adjustment — they completely
dominate the PnL of this strategy. The spec does not specify a cost model, so
the backtest will implement it cost-free first, then note this explicitly as a
major caveat. In real trading this strategy would need extremely tight spreads
and near-zero brokerage to be viable.

**Also note**: The strategy naturally prevents infinite churn on a single tick
because the spec says "hold till the futures price changes that you need to change
the strike." This means: only roll when the *nearest strike* changes, not on
every futures tick. The 237 figure already reflects this — it's strike changes,
not futures ticks (of which there were ~11,000+ that day).

### Step 4: First-tick timing

The strategy can only enter a straddle once it has a price for both legs.
With forward-fill, a price exists from the moment the first tick occurs.

If BANKNIFTY-I prints its first tick at 09:15:00, but the ATM call doesn't
print until 09:15:45, then for those first 45 seconds we have a futures price
but no option price — we can't fill. We need to know how large this lag is.

We check: for the opening ATM strike on a sample day, when does the first
tick appear in the CE file and the PE file?

In [ ]:
atm_at_open = int(atm_per_second.iloc[0])
expiry_str   = front_exp.strftime("%y%m%d")
date_str     = sample_dir.name.replace("NSE_", "")

print(f"Opening ATM strike: {atm_at_open}")
print(f"Front expiry: {front_exp.date()}")
print()

for opt_type in ["CE", "PE"]:
    fpath = sample_dir / "Options" / f"BANKNIFTY{expiry_str}{atm_at_open}{opt_type}.csv"
    if not fpath.exists():
        print(f"{opt_type}: FILE MISSING ✗")
        continue

    df = pd.read_csv(fpath, header=None, names=COLUMNS)
    df["datetime"] = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"],
        format="%Y%m%d %H:%M:%S"
    )
    first_tick    = df["datetime"].min()
    first_price   = df.loc[df["datetime"].idxmin(), "price"]
    total_ticks   = len(df)
    lag_seconds   = (first_tick - pd.Timestamp(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]} 09:15:00")).seconds

    print(f"{opt_type}: first tick at {first_tick.time()}  "
          f"(+{lag_seconds}s from open)  "
          f"price = {first_price}  "
          f"total ticks today = {total_ticks:,}")

### Finding: Both legs are active at the very start of the session

CE: first tick at 09:15:00 (+0 seconds), 22,434 total ticks today
PE: first tick at 09:15:00 (+0 seconds), 22,142 total ticks today

The ATM straddle is immediately liquid at open. No delay before we can enter.
22,000+ ticks per leg per day on the ATM strike = extremely active contract.
Compare this to the deep-OTM put from Section 1 which had only 4,780 ticks —
the ATM strike is roughly 5x more active.

### Step 5: Feasibility check — all 21 days

Now we run the opening-ATM check across every trading day for both underliers.
For each day we ask: does the opening ATM strike have both a CE and PE file?
When does each leg get its first tick?

If any day shows a missing file or a very large lag, that's a case to document
and handle explicitly in the backtest (e.g. skip that day, delay entry).

In [ ]:
def check_opening_straddle(day_dir: Path, underlier: str) -> dict:
    """
    For a given day and underlier, find the opening ATM strike and check
    whether both CE and PE files exist and when their first tick appears.
    """
    date_str = day_dir.name.replace("NSE_", "")
    date     = pd.Timestamp(date_str)
    session_open = pd.Timestamp(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]} 09:15:00")

    out = {
        "date": date.date(), "underlier": underlier,
        "front_expiry": None, "opening_spot": None, "atm_strike": None,
        "ce_first_tick_lag_s": None, "pe_first_tick_lag_s": None,
        "both_files_exist": False, "issue": ""
    }

    # front expiry
    row = front_week[
        (front_week["date"] == date) & (front_week["underlier"] == underlier)
    ]
    if row.empty:
        out["issue"] = "no front expiry"; return out
    front_exp = row["front_expiry"].iloc[0]
    out["front_expiry"] = front_exp.date()

    # futures 1-sec series
    try:
        fut_1sec = load_futures_1sec(day_dir, underlier)
    except FileNotFoundError:
        out["issue"] = "no futures file"; return out

    open_price = fut_1sec.dropna().iloc[0]
    out["opening_spot"] = round(open_price, 2)

    # available strikes
    day_u = universe[
        (universe["date"] == date) &
        (universe["underlier"] == underlier) &
        (universe["expiry_date"] == front_exp)
    ]
    strikes = np.array(sorted(day_u["strike"].unique()))
    if len(strikes) == 0:
        out["issue"] = "no strikes in universe"; return out

    atm = int(find_atm_vectorized(np.array([open_price]), strikes)[0])
    out["atm_strike"] = atm

    # check CE and PE
    expiry_str = front_exp.strftime("%y%m%d")
    missing = []
    for opt_type in ["CE", "PE"]:
        fpath = day_dir / "Options" / f"{underlier}{expiry_str}{atm}{opt_type}.csv"
        if not fpath.exists():
            missing.append(opt_type); continue

        df = pd.read_csv(fpath, header=None, names=COLUMNS)
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + " " + df["time"],
            format="%Y%m%d %H:%M:%S"
        )
        lag = (df["datetime"].min() - session_open).seconds
        out[f"{opt_type.lower()}_first_tick_lag_s"] = lag

    if missing:
        out["issue"] = f"missing files: {missing}"
    else:
        out["both_files_exist"] = True

    return out


rows = []
for day_dir in trading_days:
    for underlier in ["NIFTY", "BANKNIFTY"]:
        rows.append(check_opening_straddle(day_dir, underlier))

feas = pd.DataFrame(rows)

print("Opening ATM straddle feasibility — all 21 days × 2 underliers:\n")
print(feas.to_string(index=False))
print(f"\n✓ Both files exist : {feas['both_files_exist'].sum()}/{len(feas)}")
print(f"✗ Problem days     : {(~feas['both_files_exist']).sum()}/{len(feas)}")
print(f"\nCE first-tick lag (seconds): min={feas['ce_first_tick_lag_s'].min()}, "
      f"max={feas['ce_first_tick_lag_s'].max()}, "
      f"mean={feas['ce_first_tick_lag_s'].mean():.1f}")
print(f"PE first-tick lag (seconds): min={feas['pe_first_tick_lag_s'].min()}, "
      f"max={feas['pe_first_tick_lag_s'].max()}, "
      f"mean={feas['pe_first_tick_lag_s'].mean():.1f}")

### Finding: 42/42 days fully feasible — Phase 1 is complete

Every single (day, underlier) combination shows:
- Both CE and PE files exist ✓
- First tick within 0–2 seconds of open ✓
- Clean futures reference price with zero gaps ✓

Max first-tick lag across the entire month: 2 seconds.
Mean lag: 0.1s for CE, 0.3s for PE. Effectively immediate.

**Phase 1 conclusion — what we now know:**

| Question | Answer |
|---|---|
| Data format | Tick data, no headers, 5 columns |
| Time resolution for backtest | 1-second, using last-known price (ffill) |
| Session coverage | 09:15:00 → 15:29:59, 22,500 seconds, zero gaps |
| Trading calendar | 21 days, Nov 1–30, Nov 8 holiday |
| Front-week expiry logic | min(expiry_date >= date), rolls Thursdays |
| Underliers in scope | NIFTY + BANKNIFTY (MIDCPNIFTY ignored) |
| ATM feasibility | 42/42 day-underlier pairs fully feasible |
| Entry lag | ≤ 2 seconds in the worst case |
| Roll frequency | ~237 rolls/day on a sample BANKNIFTY day |
| Transaction cost sensitivity | CRITICAL — ~948 transactions/day/underlier |